In [3]:
import pymupdf
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load PDF
doc = pymupdf.open(
    r"C:\Users\ASUS\OneDrive\Desktop\RAG-Document-Intelligence\data\sample.pdf"
)

pages = []

for page_number, page in enumerate(doc):
    pages.append({
        "page": page_number + 1,
        "text": page.get_text()
    })


# Create chunks
def create_chunks(pages, chunk_size=1000, overlap=200):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size

            chunks.append({
                "text": text[start:end],
                "page": page["page"]
            })

            start += chunk_size - overlap

    return chunks


chunks = create_chunks(pages)

print("Chunks:", len(chunks))


# Generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts)

print("Embeddings:", embeddings.shape)

C:\Users\ASUS\anaconda3\envs\rag-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 145


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2848.88it/s]


Embeddings: (145, 384)


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings.astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 145
